# Etapa 5j — Umbral con histéresis sobre la probabilidad

**Ronda de mejoras para la entrega final (desvío 14.11 del plan).** Solo desarrollo, la misma validación cruzada por personas. El conjunto de prueba no se usa.

**Motivo.** El algoritmo corta la probabilidad de la regresión logística con un solo valor fijo (≈ 0,87). Los vasos finos tienen probabilidades más bajas que los gruesos y quedan cortados en pedazos. La histéresis (Canny, 1986) usa dos valores: lo que supera el alto es vaso seguro, y lo que supera el bajo es vaso solo si está conectado a un vaso seguro. Condurache y Mertins (2012) aplican esta regla a la salida de un clasificador lineal para segmentar vasos de retina, que es nuestro esquema. En `05c` se probó sobre el detector de líneas solo (con fracciones de píxeles): +1,2 puntos en vasos finos, con Matthews igual, y no se adoptó por simplicidad. Acá se prueba sobre la probabilidad de la combinación.

**Riesgo esperado.** En retinopatía, la mayoría de los falsos positivos están pegados a los vasos, por diferencia de ancho con lo marcado (`05h`). Bajar el umbral al lado de un vaso seguro lo ensancha. Que la ganancia en vasos finos compense ese ensanchamiento solo se puede saber midiendo.

**Candidatas** (sobre el algoritmo actual: división + relleno de exudados, 28 mapas, entrenamiento equilibrado por grosor; la probabilidad se agranda al tamaño original y se corta ahí):
- **A.** Valor fijo elegido con `choose_value`, sobre la muestra de píxeles (el algoritmo actual).
- **A′.** Valor fijo elegido como la histéresis: el de mayor Matthews medio en las imágenes enteras de entrenamiento, a resolución original, en la grilla 0,70 a 0,99 de a 0,01. Es el control para separar el efecto de la histéresis del de elegir el umbral con imágenes enteras.
- **B.** Histéresis (`segment.segmentar_histeresis_valor`, conexión de ocho vecinos): valor alto en la misma grilla y valor bajo de 0,30 a 0,95 de a 0,05, con bajo ≤ alto (bajo = alto es un valor fijo, así que B puede volver al valor fijo en una vuelta). Se elige el par con mayor Matthews medio en las imágenes enteras de entrenamiento (`segment.conteos_histeresis`).

**Medidas**, en las personas de afuera, a resolución original, en el campo de visión de la base: Matthews, Dice de línea central, vasos finos (sensibilidad de línea central) y precisión. Descriptivo: por grupo y qué agrega la histéresis (cuántos píxeles y qué fracción es vaso).

**Regla de adopción (fijada antes de correr).**
1. B reemplaza a A si (a) mejora Matthews con un intervalo al 95 % que no incluye el cero, (b) no es inferior a A en vasos finos ni en Dice de línea central (límite inferior > −0,03) y (c) supera también a A′ en Matthews con un intervalo que no incluye el cero: así la ganancia es de la histéresis y no de elegir el umbral con imágenes enteras.
2. Si B no cumple, A′ reemplaza a A si cumple (a) y (b) contra A.
3. Si ninguna cumple, queda A.

**Control:** A tiene que reproducir el Matthews de B en `05i` (0,7533).

In [ ]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.data import dev_ids, load_split
from src.evaluation import map_images
from src.pipeline import choose_value, fit_weights
from src.segment import matthews_conteos
from src.split import person_folds
from src.stats import percentile_interval, person_bootstrap_means

pd.set_option("display.width", 220)
ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))
MODE = C.SEGMENT_BACKGROUND_MODE
ALTOS = np.round(np.arange(0.70, 0.995, 0.01), 2)
BAJOS = np.round(np.arange(0.30, 0.96, 0.05), 2)
ORDER = ["A. valor fijo (muestra)", "A′. valor fijo (imágenes enteras)", "B. histéresis"]
CACHE = ROOT / "results" / "cache" / "05j_prob"
CACHE.mkdir(parents=True, exist_ok=True)
WORKERS = 4
print(MODE, "|", len(C.SEGMENT_FEATURES), "mapas |", len(ALTOS), "altos ×", len(BAJOS), "bajos")

## Pasada 1: píxeles de entrenamiento y de elección del valor fijo A

In [ ]:
def pass1(image_id):
    from src.pipeline import training_pixels
    return image_id, training_pixels(image_id, background=MODE)

t = time.time()
samples = dict(map_images(pass1, ids, workers=WORKERS))
print(f"Pasada 1: {time.time() - t:.0f} s")

## Ajuste por vuelta

In [ ]:
fit = {}
for k in range(5):
    tr = list(folds.index[folds != k])
    w = fit_weights(samples, tr)
    fit[k] = (w, choose_value(samples, w, tr))
pd.Series({f"vuelta {k}": fit[k][1] for k in range(5)}, name="valor fijo A").round(4)

## Pasada 2: probabilidades con los pesos de cada vuelta y cuentas de la grilla en entrenamiento

In [ ]:
def measures(m, truth, fov, thin):
    from src.metrics import centerline_dice, matthews
    marked = m & fov
    return {"matthews": matthews(m, truth, fov), "dice_linea_central": centerline_dice(m, truth, fov),
            "finos": m[thin].mean(), "precision": (marked & truth).sum() / max(marked.sum(), 1)}


def truth_of(image_id):
    from src.config import WIDTH_LIMITS_PX
    from src.data import load_fov, load_vessels
    from src.vessels import skeleton_widths
    truth, fov = load_vessels(image_id), load_fov(image_id)
    skel, wd = skeleton_widths(truth & fov)
    return truth, fov, skel & (wd <= WIDTH_LIMITS_PX[0])


def pass2(image_id):
    """Probabilidad con los pesos de las 5 vueltas (los mapas se calculan una vez). En las vueltas
    en que la imagen es de entrenamiento: cuentas de la grilla. En la suya: guarda la probabilidad
    y mide A."""
    import numpy as np
    from src.data import load_image
    from src.enhance import upscale
    from src.fov import detect_fov_robust
    from src.pipeline import feature_scores
    from src.segment import conteos_histeresis
    own = int(folds[image_id])
    rgb = load_image(image_id)
    truth, fov, thin = truth_of(image_id)
    auto = detect_fov_robust(rgb)
    probs, _, _ = feature_scores(rgb, [fit[k][0] for k in range(5)], 2, background=MODE)
    counts = {}
    for k, p in enumerate(probs):
        if k == own:
            np.save(CACHE / f"{image_id}.npy", p.astype(np.float32))
            row = {"image": image_id, "candidata": ORDER[0],
                   **measures((upscale(p, fov.shape) > fit[k][1]) & auto, truth, fov, thin)}
        else:
            counts[k] = conteos_histeresis(upscale(p, fov.shape), auto, truth, fov, ALTOS, BAJOS)
    return image_id, row, counts

t = time.time()
P2 = map_images(pass2, ids, workers=WORKERS)
rows_a = [r for _, r, _ in P2]
counts = {i: c for i, _, c in P2}
del P2
print(f"Pasada 2: {time.time() - t:.0f} s")

## Elección en entrenamiento (imágenes enteras, resolución original)

In [ ]:
choice, train_mcc = {}, []
for k in range(5):
    tr = list(folds.index[folds != k])
    fijo = np.mean([matthews_conteos(counts[i][k]["tp_fijo"], counts[i][k]["marcados_fijo"],
                                     counts[i][k]["positivos"], counts[i][k]["total"]) for i in tr], axis=0)
    hist = np.mean([matthews_conteos(counts[i][k]["tp"], counts[i][k]["marcados"],
                                     counts[i][k]["positivos"], counts[i][k]["total"]) for i in tr], axis=0)
    a = int(np.nanargmax(fijo))
    b, h = np.unravel_index(np.nanargmax(hist), hist.shape)
    choice[k] = {"fijo": float(ALTOS[a]), "alto": float(ALTOS[h]), "bajo": float(BAJOS[b])}
    train_mcc.append({"vuelta": k, "A (valor)": fit[k][1], "A′ (valor)": ALTOS[a], "B alto": ALTOS[h],
                      "B bajo": BAJOS[b], "Matthews entren. A′": fijo[a], "Matthews entren. B": hist[b, h]})
pd.DataFrame(train_mcc).set_index("vuelta").round(4)

## Pasada 3: máscaras de A′ y B en las personas de afuera

In [ ]:
def pass3(image_id):
    import numpy as np
    from src.data import load_image
    from src.enhance import upscale
    from src.fov import detect_fov_robust
    from src.segment import segmentar_histeresis_valor
    k = int(folds[image_id])
    truth, fov, thin = truth_of(image_id)
    auto = detect_fov_robust(load_image(image_id))
    p = upscale(np.load(CACHE / f"{image_id}.npy"), fov.shape)
    ch = choice[k]
    fijo = (p > ch["fijo"]) & auto
    hist = segmentar_histeresis_valor(p, auto, ch["alto"], ch["bajo"])
    added = hist & ~((p > ch["alto"]) & auto) & fov
    return [{"image": image_id, "candidata": ORDER[1], **measures(fijo, truth, fov, thin)},
            {"image": image_id, "candidata": ORDER[2], **measures(hist, truth, fov, thin),
             "agregado_pct_campo": 100 * added.sum() / fov.sum(),
             "agregado_es_vaso": (added & truth).sum() / max(added.sum(), 1)}]

t = time.time()
R = pd.DataFrame(rows_a + [r for rows in map_images(pass3, ids, workers=WORKERS) for r in rows])
R.to_csv(ROOT / "results" / "05j_histeresis.csv", index=False)
print(f"Pasada 3: {time.time() - t:.0f} s")

## Resultados

In [ ]:
def compare(new, old):
    out = []
    for meas in ("matthews", "dice_linea_central", "finos", "precision"):
        w = R.pivot(index="image", columns="candidata", values=meas)
        d = (w[new] - w[old]).to_frame()
        lo, hi = percentile_interval(person_bootstrap_means(d, dev["person"]))
        out.append({"medida": meas, old[:2]: w[old].mean(), new[:2]: w[new].mean(), "diferencia": d.iloc[:, 0].mean(),
                    "inferior": lo[0], "superior": hi[0], "imágenes que mejoran": int((d.iloc[:, 0] > 0).sum())})
    return pd.DataFrame(out).set_index("medida")

tables = {"B − A": compare(ORDER[2], ORDER[0]), "A′ − A": compare(ORDER[1], ORDER[0]),
          "B − A′": compare(ORDER[2], ORDER[1])}
for name, t in tables.items():
    print(name); display(t.round(4))

In [ ]:
def meets(t):
    return bool(t.loc["matthews", "inferior"] > 0 and t.loc["finos", "inferior"] > -C.NONINF_MARGIN_MATTHEWS
                and t.loc["dice_linea_central", "inferior"] > -C.NONINF_MARGIN_MATTHEWS)

b_ok = meets(tables["B − A"]) and tables["B − A′"].loc["matthews", "inferior"] > 0
a2_ok = meets(tables["A′ − A"])
adopt = ORDER[2] if b_ok else ORDER[1] if a2_ok else ORDER[0]
print("B cumple (a)(b):", meets(tables["B − A"]), "| B supera a A′ (c):",
      bool(tables["B − A′"].loc["matthews", "inferior"] > 0), "| A′ cumple (a)(b):", a2_ok)
print("→ se adopta:", adopt)

In [ ]:
print("Por grupo (media):")
display(R.join(dev["group"], on="image").pivot_table(index="candidata", columns="group",
        values=["matthews", "precision", "finos"]).loc[ORDER].round(3))
print("Qué agrega la histéresis (B, personas de afuera):")
R[R["candidata"] == ORDER[2]].join(dev["group"], on="image").groupby("group")[
    ["agregado_pct_campo", "agregado_es_vaso"]].mean().round(3)

## Decisión (escrita después de correr)

*(pendiente)*